# FastAPI Backend Development

The machine learning pipeline developed in previous notebooks will now be exposed as a REST API using FastAPI.

FastAPI is a modern Python web framework widely used for deploying machine learning models in production environments.

The objectives of this notebook are:

- Load trained models
- Build API request and response schemas
- Expose prediction endpoints
- Validate model inference through API calls

This notebook serves as the foundation for future dashboard integration and deployment.

## Required Libraries

FastAPI will be used to expose model predictions through REST endpoints.

Pydantic will be used for request validation and data serialization.

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
import pandas as pd
import numpy as np
import joblib

## Loading Trained Models

The production models and feature schema are loaded from disk to support API-based inference.

In [ ]:
loaded_lgbm = joblib.load(
    "../models/lightgbm_final.pkl"
)

loaded_xgb = joblib.load(
    "../models/xgboost_final.pkl"
)

feature_columns = joblib.load(
    "../models/feature_columns.pkl"
)

print("Models Loaded Successfully")
print(f"Features: {len(feature_columns)}")

In [ ]:
def generate_risk_score(probability):

    return round(probability * 100, 2)

In [ ]:
def get_risk_category(
    probability,
    strategy="balanced"
):

    strategy_thresholds = {
        "risk_sensitive": 0.50,
        "balanced": 0.55,
        "performance_optimized": 0.60
    }

    threshold = strategy_thresholds[strategy]

    if probability >= threshold:
        return "High Risk"

    elif probability >= threshold * 0.6:
        return "Medium Risk"

    else:
        return "Low Risk"

In [ ]:
def predict_customer_risk(
    customer_data,
    strategy="balanced"
):

    lgbm_prob = loaded_lgbm.predict_proba(
        customer_data
    )[:, 1][0]

    xgb_prob = loaded_xgb.predict_proba(
        customer_data
    )[:, 1][0]

    probability = (
        lgbm_prob + xgb_prob
    ) / 2

    risk_score = generate_risk_score(
        probability
    )

    risk_category = get_risk_category(
        probability,
        strategy
    )

    return {
        "probability": round(
            float(probability),
            4
        ),
        "risk_score": float(risk_score),
        "risk_category": risk_category,
        "strategy": strategy
    }

## Creating the FastAPI Application

A FastAPI application instance is created to expose machine learning predictions through REST endpoints.

The API will later be consumed by the React frontend and can also be tested directly through the automatically generated Swagger UI.

In [ ]:
app = FastAPI(
    title="CreditWise AI",
    description="Loan Default Risk Prediction API",
    version="1.0"
)

## Request Schema

Pydantic models are used to validate incoming requests and ensure that API consumers provide data in the expected format.

In [ ]:
class PredictionRequest(BaseModel):
    
    strategy: str = "balanced"
    
    features: dict

## Response Schema

The API returns a structured response containing the predicted probability of default, risk score, risk category, and selected lending strategy.

In [ ]:
class PredictionResponse(BaseModel):

    probability: float

    risk_score: float

    risk_category: str

    strategy: str

## Health Check Endpoint

A health check endpoint is implemented to verify that the API service is running and available.

In [ ]:
@app.get("/")
def home():

    return {
        "message": "CreditWise AI API Running"
    }

## Prediction Endpoint

The prediction endpoint accepts customer feature values and a selected lending strategy.

Incoming features are converted into a DataFrame matching the model's training schema before being passed through the ensemble prediction pipeline.

The endpoint returns:

- Default probability
- Risk score
- Risk category
- Selected strategy

In [ ]:
@app.post(
    "/predict",
    response_model=PredictionResponse
)
def predict(
    request: PredictionRequest
):

    customer_df = pd.DataFrame(
        [request.features]
    )

    customer_df = customer_df.reindex(
        columns=feature_columns,
        fill_value=0
    )

    result = predict_customer_risk(
        customer_df,
        strategy=request.strategy
    )

    return result

## Endpoint Logic Validation

Before exposing the API through FastAPI, the endpoint logic is validated using a sample customer record from the dataset.

In [ ]:
train_df = pd.read_csv(
    "../data/processed/train_processed.csv"
)

X = train_df.drop(
    columns=["TARGET"]
)

X = X.reindex(
    columns=feature_columns,
    fill_value=0
)

print(X.shape)

In [ ]:
sample_features = (
    X.iloc[0]
    .to_dict()
)

sample_request = PredictionRequest(
    strategy="balanced",
    features=sample_features
)

predict(sample_request)

## Production Transition

The API logic developed in this notebook will ultimately be moved into a standalone FastAPI application file.

This separation allows the model to be deployed independently from the notebook environment and consumed by external applications.

In [ ]:
import os

os.makedirs(
    "../src/api",
    exist_ok=True
)

print("API folder ready")

## Production FastAPI Application

The validated API logic is now migrated into a standalone FastAPI application file.

This application can be executed independently from the notebook environment and serves as the production backend for CreditWise AI.

In [ ]:
main_py_content = '''
from pathlib import Path
from fastapi import FastAPI
from pydantic import BaseModel
import pandas as pd
import joblib

# Load models
BASE_DIR = Path(__file__).resolve().parent.parent.parent

MODELS_DIR = BASE_DIR / "models"

loaded_lgbm = joblib.load(
    MODELS_DIR / "lightgbm_final.pkl"
)

loaded_xgb = joblib.load(
    MODELS_DIR / "xgboost_final.pkl"
)

feature_columns = joblib.load(
    MODELS_DIR / "feature_columns.pkl"
)

app = FastAPI(
    title="CreditWise AI",
    description="Loan Default Risk Prediction API",
    version="1.0"
)

class PredictionRequest(BaseModel):
    strategy: str = "balanced"
    features: dict

class PredictionResponse(BaseModel):
    probability: float
    risk_score: float
    risk_category: str
    strategy: str

def generate_risk_score(probability):
    return round(probability * 100, 2)

def get_risk_category(probability, strategy="balanced"):

    strategy_thresholds = {
        "risk_sensitive": 0.50,
        "balanced": 0.55,
        "performance_optimized": 0.60
    }

    threshold = strategy_thresholds[strategy]

    if probability >= threshold:
        return "High Risk"
    elif probability >= threshold * 0.6:
        return "Medium Risk"
    else:
        return "Low Risk"

def predict_customer_risk(customer_data, strategy="balanced"):

    lgbm_prob = loaded_lgbm.predict_proba(
        customer_data
    )[:, 1][0]

    xgb_prob = loaded_xgb.predict_proba(
        customer_data
    )[:, 1][0]

    probability = (
        lgbm_prob + xgb_prob
    ) / 2

    risk_score = generate_risk_score(
        probability
    )

    risk_category = get_risk_category(
        probability,
        strategy
    )

    return {
        "probability": round(float(probability), 4),
        "risk_score": float(risk_score),
        "risk_category": risk_category,
        "strategy": strategy
    }

@app.get("/")
def home():
    return {
        "message": "CreditWise AI API Running"
    }

@app.post(
    "/predict",
    response_model=PredictionResponse
)
def predict(request: PredictionRequest):

    customer_df = pd.DataFrame(
        [request.features]
    )

    customer_df = customer_df.reindex(
        columns=feature_columns,
        fill_value=0
    )

    return predict_customer_risk(
        customer_df,
        strategy=request.strategy
    )
'''

with open(
    "../src/api/main.py",
    "w",
    encoding="utf-8"
) as f:
    f.write(main_py_content)

print("main.py created successfully")

In [ ]:
#payload testing
sample_features = (
    X.iloc[0]
    .to_dict()
)

In [ ]:
import json

payload = {
    "strategy": "balanced",
    "features": sample_features
}

print(
    json.dumps(
        payload,
        indent=2
    )
)